# MAPED: experimental data on CPU

Load your tilt series, align it step by step, and inspect the merged patterns.

## Setup

**Development prerequisite:** this workflow still requires MAPED development builds; the package command alone does not yet provide these APIs.

New to Python? Follow the [Conda or uv setup guide](https://github.com/bobleesj/quantem-tutorials/blob/oct-17-maped-tutorials/tutorials/diffraction/maped/setup/README.md). If packages are missing in an existing Python 3.11+ environment, uncomment the cell below, then restart the kernel.

In [ ]:
# %pip install --upgrade quantem pandas

In [ ]:
from pathlib import Path

from quantem.core.datastructures import Dataset4dstem
from quantem.diffraction import MAPED

folder = Path("cropped-tilts")
output = Path("outputs/merged_cpu.zip")

## Load data

Use small, already-cropped and detector-corrected Arina recordings, for example 64 × 64 scan positions. Keep their master, data, and metadata files together. The native reader loads each complete recording; cropping after loading does not reduce that initial memory use. This notebook uses native QuantEM throughout and does not import GPU or widget packages.

In [ ]:
files = sorted(folder.glob("*_master.h5"))
tilts = [Dataset4dstem.from_file(file, "arina") for file in files]
maped = MAPED.from_datasets(tilts)
maped.show_tilts(num_cols=7, size="medium")

## Align

`align_diffraction()` measures detector shifts; `align_images()` measures scan shifts from the default whole-detector mean images. Shifts use `(row, col)` pixels.

To use BF, ADF, DF or simultaneously recorded HAADF images, pass `images=images` to `align_images()`: one 2D array per tilt, with the same scan shape, sampling and orientation. The merge still uses the original diffraction data.

In [ ]:
maped.find_beam_center(show=False)
maped.align_diffraction(show=False)
maped.align_images(iterations=160, show=False)

In [ ]:
maped.show_alignment(num_cols=7, size="medium", reference=0)  # Set the zero-tilt index.

## Merge

Keep the common field of view and preserve the computed float32 values.

In [ ]:
merged = maped.merge(crop=True, show=False)
maped.report()  # Table; use as_dict=True for a dictionary.

## Inspect results

Compare individual diffraction patterns before and after merging at three marked scan positions. Choose positions inside your scan; the examples below assume 64 × 64 input scans.

In [ ]:
maped.show(positions=[(15, 15), (25, 25), (40, 40)], tilt_indices=[0])

## Save and reopen (optional)

Native QuantEM ZIP serialization preserves the merged values and alignment metadata.

In [ ]:
# from quantem.core.io import load
# saved = output
# saved.parent.mkdir(parents=True, exist_ok=True)
# merged.save(saved)
# reopened = load(saved)
# reopened

Built on Henry Bell’s [QuantEM PR #256](https://github.com/electronmicroscopy/quantem/pull/256). Experimental inputs and executed outputs are not included in this shareable template.